# TREXIO Converter

*Lucas Visscher (Vrije Universiteit Amsterdam) & Anthony Scemama (CNRS Toulouse) 2025*

Dependencies:
 - h5py library (`conda install h5py`)
 - trexio library (`pip install trexio`)

Since release DIRAC23 all data suitable for restart and analysis purposes is stored on the file CHECKPOINT.h5

This file is in hdf5 format allowing for easy export of the generated data to other formats. The structure is
given by the data schema contained in the file `utils/DIRACschema.txt` and can be inspected by h5dump and other hdf5 tools.
Please make sure the checkpoint file (renamed and exported by the pam run script of DIRAC) is contained in the directory in which you start this Jupyter Notebook before running this code. 

In [ ]:
# Check dependencies
try:
    import trexio
except ImportError:
    raise Exception("Unable to import trexio. Please check that trexio is properly installed.")

try:
    import h5py
except ImportError:
    raise Exception("Unable to import h5py. Please check that h5py is properly installed.")

try:
    import numpy as np
except ImportError:
    raise Exception("Unable to import numpy. Please check that numpy is properly installed.")

In [ ]:
# Utility functions to read hdf5 into a dictionary

def recursively_load_dict_contents_from_group(h5file, data_dict, path):
    """
    Get a flat dictionary with all data
    """
    for key, item in h5file[path].items():
        if isinstance(item, h5py._hl.dataset.Dataset):
            data_dict[path+key] = item[()]
        elif isinstance(item, h5py._hl.group.Group):
            recursively_load_dict_contents_from_group(h5file, data_dict, path + key + '/')
    return

def read_hdf5(file_name):
    """
    Open hdf5-type file and return dictionary of its contents
    """
    data_dict = {}
    with h5py.File(file_name, 'r') as h5file:
        recursively_load_dict_contents_from_group(h5file, data_dict,'/')
    return data_dict

In [ ]:
# Define and read the DIRAC file you wish to convert, the trexio file will get the same name with trexio inserted
dirac_file = 'scf_h2o.h5'
data = read_hdf5(dirac_file)

In the following cell we list the large component basis together with its expansion centers and exponential parameters after first defining the proper symbols (s, p, d, f, g) to indicate the functions. We also use numpy to reshape the data for ease of printing. In DIRAC all arrays are always stored as 1-dimensional while they are usually conceptually multidimensional.

In [ ]:
name_l = ['-','s','p','d','f','g','h','i']

# Extract the large component basis, this is aobasis #1.
n_shells = data['/input/aobasis/1/n_shells'][0]
orbmom = data['/input/aobasis/1/orbmom']
center = np.array(data['/input/aobasis/1/center']).reshape((n_shells,3))
exponents = np.array(data['/input/aobasis/1/exponents'])
contractions = np.array(data['/input/aobasis/1/contractions'])

# Print the large component basis (formatting could be done more elegantly...).
print ('     Center           Type    Exponent Contraction\n',49*'-')
offset = 0
for shell, lvalue  in enumerate(orbmom):
    n_prim = data['/input/aobasis/1/n_prim'][shell]
    format_string = '{:6.3f} {:6.3f} {:6.3f}    {}   {:>10.2e} {:>10.2e}'
    print (format_string.format(*center[shell,0:3],name_l[lvalue],exponents[offset],contractions[offset]))
    format_string = 28*' '+'{:>10.2e} {:>10.2e}'
    for i in range(offset+1,offset+n_prim):
        print (format_string.format(exponents[i],contractions[i]))
    offset += n_prim

Note that each shell has its own center defined, this is done for simplicity and ease of resorting basis sets. Note also that the coordinates can be different from the input values as DIRAC will by default reorient the molecule such that the center of mass is in the origin. This is also visible in the coordinates of the atoms of this molecule that are updated and can be extracted also from the input section. Note that these are stored in Ångstrom, while the expansion center coordinates are stored in Bohrs (atomic units).

In [ ]:
# Print the molecular coordinates
print (data['/input/molecule/geometry'])

Turning to the results section we will print first the orbital energies obtained in the SCF. These are stored in atomic units (Hartrees in this case) as well.

In [ ]:
# Print the orbital energies
print (data['/result/wavefunctions/scf/mobasis/eigenvalues'])

Other data can be extracted in a similar way, please consult file `DIRACschema.txt` for concise definitions of all the data that is stored.

## Converting to TREXIO

In [ ]:
filename = dirac_file.replace('.h5','_trexio.h5')

import os
try:
    os.remove(filename)
except:
    print(f"File {filename} does not yet exist.")

# Open trexio file
trexio_file = trexio.File(filename, mode='w', back_end=trexio.TREXIO_HDF5)

In [ ]:
# Write nucleus group
xtang2022 = 0.529177210544 # Bohr to angstrom value used in DIRAC
angstrom  = 1 / xtang2022
nucleus_label = [ label.decode('UTF8') for label in data['/input/molecule/symbols'] ]

trexio.write_nucleus_num(trexio_file,int(data['/input/molecule/n_atoms'][0]))
trexio.write_nucleus_charge(trexio_file,[data['/input/molecule/nuc_charge']])
trexio.write_nucleus_label(trexio_file,nucleus_label)
trexio.write_nucleus_coord(trexio_file,np.array(data['/input/molecule/geometry']) * angstrom)

In [ ]:
# Write electrons group
up_num = int(np.sum(data['/result/wavefunctions/scf/mobasis/occupations']))
dn_num = up_num # DIRAC is always restricted
elec_num = up_num + dn_num
trexio.write_electron_num(trexio_file, elec_num)
trexio.write_electron_up_num(trexio_file, up_num)
trexio.write_electron_dn_num(trexio_file, dn_num)

In [ ]:
# Write basis set data

# Setup dictionary of angular momenta
l_dict = {}
for l,key in enumerate("spdfghijkl"):
    l_dict[key] = l

coord = trexio.read_nucleus_coord(trexio_file)
    
shell_num = data['/input/aobasis/1/n_shells'][0]
nucleus_index = []
shell_ang_mom = []
shell_factor = [1. for _ in range(shell_num)]  # 1 for cartesians. Will be set in AOs
r_power = [0 for _ in range(shell_num)]        # Always zero for Gaussians
shell_index = []
exponent = []
coefficient = []
prim_factor = []
offset = 0
for shell, lvalue  in enumerate(orbmom):
    shell_index.append(shell)

    # Find index of the center
    v = center[shell,0:3] - coord
    distances = [np.linalg.norm(u) for u in v]
    index = np.argmin(distances)
    nucleus_index.append(index)
    shell_ang_mom.append(l_dict[name_l[lvalue]])

    n_prim = data['/input/aobasis/1/n_prim'][shell]
    #for i in range(offset+1,offset+n_prim):
    for i in range(offset,offset+n_prim):
        exponent.append(exponents[i])
        coefficient.append(contractions[i])
#       Compute here the normalization factor of the primitive
        prim_factor.append(1.) # DIRAC has the normalization included in the contraction coefficients, so can just put 1 here.
    offset += n_prim    
prim_num = offset

# Trexio needs offsets to find the right shell for the primitive list
offset = 0
shell_index = []
for shell, n_cont  in enumerate(data['/input/aobasis/1/n_cont']):
    n_prim = data['/input/aobasis/1/n_prim'][shell]
    for i in range(offset,offset+n_prim):
        shell_index.append(offset)
    offset += 1

trexio.write_basis_type(trexio_file, "Gaussian")
trexio.write_basis_prim_num(trexio_file, int(prim_num))
trexio.write_basis_shell_num(trexio_file,int(shell_num))
trexio.write_basis_nucleus_index(trexio_file, nucleus_index)
trexio.write_basis_shell_ang_mom(trexio_file, shell_ang_mom)
trexio.write_basis_shell_factor(trexio_file, shell_factor)
trexio.write_basis_shell_index(trexio_file, shell_index)
trexio.write_basis_r_power(trexio_file, r_power)
trexio.write_basis_exponent(trexio_file,exponent)
trexio.write_basis_coefficient(trexio_file,coefficient)
trexio.write_basis_prim_factor(trexio_file,prim_factor)

ao_num = data['/input/aobasis/1/n_ao'][0]
ao_shell = []
ao_cartesian = 1
ao_normalization = [ 1. for i in range(ao_num) ]

if ao_cartesian:
    for ishell,l in enumerate(shell_ang_mom):
        for _ in range((l+1)*(l+2)//2):
            ao_shell.append(ishell)
else: # Shperical harmonics
    for ishell,l in enumerate(shell_ang_mom):
        for _ in range(-l,l):
            ao_shell.append(ishell)
            
trexio.write_ao_cartesian(trexio_file,ao_cartesian)
trexio.write_ao_num(trexio_file, int(ao_num))
trexio.write_ao_shell(trexio_file,ao_shell)
trexio.write_ao_normalization(trexio_file, ao_normalization)

In [ ]:
# Write MO coefficients
mo_num = int(data['/result/wavefunctions/scf/mobasis/n_mo'][0])
trexio.write_mo_type(trexio_file,"Canonical")
trexio.write_mo_num(trexio_file,mo_num)
trexio.write_mo_energy(trexio_file,[data['/result/wavefunctions/scf/mobasis/eigenvalues']])
trexio.write_mo_occupation(trexio_file,[2*data['/result/wavefunctions/scf/mobasis/occupations']])
trexio.write_mo_coefficient(trexio_file,[data['/result/wavefunctions/scf/mobasis/orbitals'][0:ao_num*mo_num]])

In [ ]:
# Utility routine (written by copilot) to unpack matrices
def unpack_symmetric_matrix(packed, n):
    """
    Unpacks a 1D array of size n(n+1)/2 into a symmetric n x n matrix.
    Assumes the packed array contains the lower triangle (row-wise).
    """
    if len(packed) != n * (n + 1) // 2:
        raise ValueError('Packed array size does not match n(n+1)/2.')

    matrix = np.zeros((n, n))
    tril_indices = np.tril_indices(n)
    matrix[tril_indices] = packed
    matrix = matrix + np.tril(matrix, -1).T # Mirror lower triangle to upper
    return matrix

In [ ]:
# Write overlap
overlap = unpack_symmetric_matrix(data['/result/operators/ao_matrices/OVERLAP TFFT'],ao_num)
# Write AO integrals
trexio.write_ao_1e_int_overlap(trexio_file,overlap)

In [ ]:
# Close the file
trexio_file.close()

In [ ]:
# Check reading from the file
trexio_file = trexio.File(filename, mode='r', back_end=trexio.TREXIO_HDF5)
coords_r = trexio.read_nucleus_coord(trexio_file)
print(f"Nuclear coordinates from {filename} TREXIO file: \n{coords_r}")
coords_r = trexio.read_nucleus_coord(trexio_file)
electron_num = trexio.read_electron_num(trexio_file)
z_num = trexio.read_nucleus_charge(trexio_file)
print(f"Charge of the molecule deduced from information on this file: {float(np.sum(z_num))-electron_num}")